# Siamese Networks for Few-Shot Image Classification with Explainable AI (XAI)

## 📋 Research Paper-Level Documentation

### Abstract
This notebook implements a Siamese Network-based few-shot learning system integrated with Explainable AI (XAI) techniques for medical image classification. Unlike prototypical networks that compute class prototypes, Siamese networks learn to compare pairs of images and classify based on similarity to support examples.

### 1. Introduction
Siamese Networks are a class of neural networks that learn to compare two inputs by encoding them into a shared embedding space. For few-shot learning, we compare each query image against support set images and determine classification based on similarity.

**Key Difference from Prototypical Networks:**
- Prototypical: Compute class prototypes, compare query to prototypes
- Siamese: Compare query to individual support images, aggregate similarities

**Problem Setup:**
- **N-way K-shot classification**: Given N classes with K support examples each
- **Support set**: Examples used for similarity comparison
- **Query set**: Examples to be classified by comparing to support

### 2. Mathematical Formulation

#### 2.1 Siamese Architecture
The Siamese architecture uses two identical encoder branches (weight sharing):

$$\mathbf{z}_1 = f_\phi(\mathbf{x}_1), \quad \mathbf{z}_2 = f_\phi(\mathbf{x}_2)$$

where $f_\phi$ is the shared encoder and $\mathbf{z}_i \in \mathbb{R}^d$ are the embeddings.

#### 2.2 Distance Metric
Squared Euclidean distance:

$$d(\mathbf{z}_1, \mathbf{z}_2) = \|\mathbf{z}_1 - \mathbf{z}_2\|^2 = \sum_{j=1}^{d}(z_{1,j} - z_{2,j})^2$$

#### 2.3 Similarity Measure
Convert distance to similarity using exponential:

$$s = \exp(-d(\mathbf{z}_1, \mathbf{z}_2))$$

#### 2.4 Contrastive Loss (Training)
For training with pairs:

$$\mathcal{L}_{contrastive} = (1-Y) \cdot D^2 + Y \cdot \max(0, m - D)^2$$

where $Y \in \{0,1\}$ indicates similar (1) or dissimilar (0) pairs, and $m$ is the margin.

#### 2.5 Classification via Similarity Aggregation
For N-way classification, aggregate similarities to each class:

$$\text{score}(c) = \sum_{i: y_i = c} s_i = \sum_{i: y_i = c} \exp(-d(\mathbf{z}_q, \mathbf{z}_i))$$

Probability:

$$P(y=c | \mathbf{x}_q) = \frac{\exp(\text{score}(c))}{\sum_{c'} \exp(\text{score}(c'))}$$

![Few-Shot Learning Architecture](few%20shot%20image%20classification.png)

**Figure: Siamese Network Architecture** - Both branches share weights. The query is compared to all support images, similarities are aggregated by class, and the query is classified to the class with highest total similarity.

In [ ]:
# ============================================================
# IMPORTS AND CONFIGURATION
# ============================================================

import os
import random
from pathlib import Path
import json
import math
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, 
    precision_recall_fscore_support, classification_report
)
from sklearn.model_selection import StratifiedShuffleSplit
from scipy.stats import ttest_ind, sem
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for publication-quality plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"{'='*60}")
print(f"Device Configuration")
print(f"{'='*60}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cudnn.benchmark = True
print(f"Using: {DEVICE}")
print(f"{'='*60}\n")

## 3. Configuration and Hyperparameters

### 3.1 Experimental Setup

We use a standardized configuration for few-shot learning experiments:

| Parameter | Value | Description |
|-----------|-------|-------------|
| N-way | 8 | Number of classes per episode |
| K-shot | 5 | Support examples per class |
| Q-query | 15 | Query examples per class |
| Embedding dim | 128 | Dimension of learned embeddings |
| Learning rate | 1e-3 | Initial learning rate |
| Weight decay | 1e-4 | L2 regularization |
| Epochs | 30 | Training iterations |
| Margin | 1.0 | Contrastive loss margin |

In [ ]:
# ============================================================
# HYPERPARAMETERS AND PATHS
# ============================================================

# Data paths - MODIFY THESE FOR YOUR DATASET
DATA_ROOT = '/kaggle/input/cucumber-dataset/Original Image'  # Change to your dataset path
OUTPUT_DIR = '/kaggle/working'

# Create output directories
SPLIT_DIR = os.path.join(OUTPUT_DIR, 'splits')
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')
XAI_DIR = os.path.join(OUTPUT_DIR, 'xai')
CKPT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')

for p in [SPLIT_DIR, PLOTS_DIR, XAI_DIR, CKPT_DIR]: 
    os.makedirs(p, exist_ok=True)

# Random seed for reproducibility
RNG_SEED = 42
torch.manual_seed(RNG_SEED)
np.random.seed(RNG_SEED)
random.seed(RNG_SEED)

# Dataset splitting ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Few-shot learning parameters
N_WAY = 8           # Number of classes per episode
K_SHOT = 5          # Support examples per class
Q_QUERY = 15        # Query examples per class

# Training parameters
EPISODES_PER_EPOCH = 20
VAL_EPISODES = 10
TEST_EPISODES = 20
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MARGIN = 1.0         # Contrastive loss margin

EMBEDDING_DIM = 128
IMAGE_SIZE = 128

# Print configuration
print(f"{'='*60}")
print(f"Experimental Configuration")
print(f"{'='*60}")
print(f"Data Root: {DATA_ROOT}")
print(f"N-way K-shot: {N_WAY}-way {K_SHOT}-shot")
print(f"Q-query: {Q_QUERY} per class")
print(f"Episodes per epoch: {EPISODES_PER_EPOCH}")
print(f"Total epochs: {NUM_EPOCHS}")
print(f"Embedding dimension: {EMBEDDING_DIM}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Contrastive margin: {MARGIN}")
print(f"{'='*60}\n")

## 4. Data Handling and Preprocessing

### 4.1 Stratified Data Splitting

**Research Context:** Proper data splitting is crucial for few-shot learning to ensure representative class distributions across train/val/test sets.

**Algorithm:**
1. Collect all images and labels from class subdirectories
2. Split into train (80%) and temp (20%) using StratifiedShuffleSplit
3. Split temp into validation (50%) and test (50%)

**Mathematical Formulation:**
For stratified splitting with $C$ classes:

$$|\mathcal{T}_{train}| = \sum_{c=1}^{C} \lfloor n_c \cdot r_{train} \rfloor$$

In [ ]:
# ============================================================
# DATA SPLITTING FUNCTIONS
# ============================================================

def make_stratified_splits(root_dir, seed=RNG_SEED):
    """
    Perform stratified train/val/test splitting.
    
    Algorithm:
    1. Collect all images and labels from class subdirectories
    2. Split into train (80%) and temp (20%) using StratifiedShuffleSplit
    3. Split temp into validation (50%) and test (50%)
    
    Returns:
        df_train, df_val, df_test: pandas DataFrames with image paths and labels
    """
    data = []
    root = Path(root_dir)
    
    # Discover class directories
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    print(f"Discovered {len(classes)} classes: {classes}")
    
    if len(classes) < 2:
        raise ValueError('Dataset root must contain at least 2 class subfolders')
    
    # Collect image paths with labels
    for lbl, cls in enumerate(classes):
        images = list((root / cls).glob('*'))
        images = [x for x in images if x.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tif']]
        for img in images:
            data.append({'image': str(img), 'label': lbl, 'class': cls})
    
    df = pd.DataFrame(data)
    print(f"Total images: {len(df)}")
    print(f"Images per class: {df['label'].value_counts().sort_index().tolist()}")
    
    x = df['image']
    y = df['label']
    
    # First split: 80% train, 20% temp
    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=(1-TRAIN_RATIO), random_state=seed)
    train_idx, temp_idx = next(splitter1.split(x, y))
    
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_temp = df.iloc[temp_idx].reset_index(drop=True)
    
    # Second split: 10% val, 10% test from temp
    test_ratio_adjusted = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=test_ratio_adjusted, random_state=seed)
    val_idx, test_idx = next(splitter2.split(df_temp['image'], df_temp['label']))
    
    df_val = df_temp.iloc[val_idx].reset_index(drop=True)
    df_test = df_temp.iloc[test_idx].reset_index(drop=True)
    
    # Save splits as CSV
    df_train.to_csv(os.path.join(SPLIT_DIR, 'train.csv'), index=False)
    df_val.to_csv(os.path.join(SPLIT_DIR, 'val.csv'), index=False)
    df_test.to_csv(os.path.join(SPLIT_DIR, 'test.csv'), index=False)
    
    print(f"\nStratified Split Results:")
    print(f"  Training:   {len(df_train)} images ({len(df_train)/len(df)*100:.1f}%)")
    print(f"  Validation: {len(df_val)} images ({len(df_val)/len(df)*100:.1f}%)")
    print(f"  Testing:    {len(df_test)} images ({len(df_test)/len(df)*100:.1f}%)")
    
    return df_train, df_val, df_test, classes

In [ ]:
# Perform data splitting
df_train, df_val, df_test, CLASS_NAMES = make_stratified_splits(DATA_ROOT)
NUM_CLASSES = len(CLASS_NAMES)
print(f"\nClass names: {CLASS_NAMES}")

### 4.2 Data Augmentation and Transforms

**Research Context:** Data augmentation prevents overfitting on limited support set examples.

**Augmentation Pipeline:**
- Random horizontal flipping (p=0.5)
- Random rotation (±15°)
- Color jitter (brightness, contrast, saturation, hue)
- ImageNet normalization

In [ ]:
# ============================================================
# DATASET AND TRANSFORMS
# ============================================================

class ImagePathsDataset(Dataset):
    """
    Custom dataset for loading images from CSV file paths.
    
    Args:
        df: DataFrame with 'image' and 'label' columns
        transform: Optional torchvision transform to apply
    """
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image']).convert('RGB')
        image = transforms.ToTensor()(image)
        
        if self.transform is not None:
            image = self.transform(image)
        
        return image, int(row['label'])


def make_transforms(img_size=IMAGE_SIZE):
    """
    Create training and evaluation transforms.
    """
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(
            brightness=0.1, 
            contrast=0.1, 
            saturation=0.1, 
            hue=0.05
        ),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, eval_transform

train_transform, eval_transform = make_transforms()

In [ ]:
# ============================================================
# SIAMESE NETWORK EPISODIC SAMPLER
# ============================================================

class SiameseSampler:
    """
    Sampler for creating few-shot episodes using Siamese pair comparison.
    
    Each episode consists of:
    - N-way classes selected from the dataset
    - K-shot support examples per class
    - Q-query test examples per class
    
    For Siamese networks, we compare each query to all support examples
    and determine the most similar class.
    """
    def __init__(self, labels, n_way, k_shot, q_query, episodes, seed=RNG_SEED):
        self.labels = np.array(labels)
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.episodes = episodes
        self.rng = np.random.RandomState(seed)
        
        # Group indices by class
        self.by_class = {c: np.where(self.labels == c)[0] for c in np.unique(self.labels)}
        
        # Verify sufficient samples
        min_required = k_shot + q_query
        for c, idx in self.by_class.items():
            if len(idx) < min_required:
                raise ValueError(
                    f'Class {c} has {len(idx)} samples, '
                    f'but needs at least {min_required} for '
                    f'k_shot={k_shot}, q_query={q_query}'
                )

    def __len__(self):
        return self.episodes

    def __iter__(self):
        for _ in range(self.episodes):
            # Select N classes without replacement
            selected_classes = self.rng.choice(
                list(self.by_class.keys()), 
                size=self.n_way, 
                replace=False
            )
            
            support_idx = []
            query_idx = []
            
            for c in selected_classes:
                # Sample K+Q examples for this class
                choices = self.rng.choice(
                    self.by_class[c], 
                    size=self.k_shot + self.q_query, 
                    replace=False
                )
                support_idx.extend(choices[:self.k_shot].tolist())
                query_idx.extend(choices[self.k_shot:].tolist())
            
            yield support_idx, query_idx


def pair_loader(df, transform, n_way=N_WAY, k_shot=K_SHOT, 
                q_query=Q_QUERY, episodes=EPISODES_PER_EPOCH):
    """Helper function to create dataset and sampler."""
    dataset = ImagePathsDataset(df, transform=transform)
    sampler = SiameseSampler(
        df['label'].to_numpy(), 
        n_way=n_way, 
        k_shot=k_shot, 
        q_query=q_query, 
        episodes=episodes
    )
    return dataset, sampler

## 5. Siamese Network Architecture

### 5.1 Convolutional Encoder

**Architecture Design:**

| Layer | Output Channels | Kernel | Output Size |
|-------|----------------|--------|-------------|
| Conv1 + BN + ReLU + Pool | 64 | 3×3 | 64×64 |
| Conv2 + BN + ReLU + Pool | 128 | 3×3 | 32×32 |
| Conv3 + BN + ReLU + Pool | 128 | 3×3 | 16×16 |
| Conv4 + BN + ReLU + Pool | 256 | 3×3 | 8×8 |
| Adaptive Avg Pool | 256 | - | 2×2 |
| FC | 256 → 128 | - | 128 |

**Design Rationale:**
- Weight sharing between two branches
- Batch normalization stabilizes training
- L2 normalization for better similarity matching
- Higher channel count (256) for richer representations

In [ ]:
# ============================================================
# SIAMESE NETWORK MODEL
# ============================================================

class SiameseEncoder(nn.Module):
    """
    Convolutional encoder for Siamese Networks.
    
    Architecture: 4 convolutional blocks with batch normalization,
    ReLU activation, and max pooling, followed by a fully connected
    layer to produce fixed-dimensional embeddings with L2 normalization.
    
    Forward pass:
    x → ConvBlock1 → ConvBlock2 → ConvBlock3 → ConvBlock4 → AdaptivePool → FC → L2Norm → z
    """
    def __init__(self, out_dim=128):
        super().__init__()
        
        # Convolutional blocks
        self.conv_blocks = nn.Sequential(
            # Block 1: 3 -> 64 channels
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 128 -> 64
            
            # Block 2: 64 -> 128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 64 -> 32
            
            # Block 3: 128 -> 128 channels
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 -> 16
            
            # Block 4: 128 -> 256 channels
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16 -> 8
            
            # Adaptive pooling to fixed size
            nn.AdaptiveAvgPool2d((2, 2))
        )
        
        # Projection head
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 2 * 2, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, out_dim)
        )
    
    def forward(self, x):
        x = self.conv_blocks(x)     # [B, 256, 2, 2]
        x = self.fc(x)              # [B, out_dim]
        # L2 normalize for better similarity matching
        x = F.normalize(x, p=2, dim=1)
        return x


class SiameseNetwork(nn.Module):
    """
    Siamese Network for few-shot classification via pairwise comparison.
    
    Components:
    1. Encoder: Shared embedding network for both branches
    2. Similarity computation: Compare query to support images
    3. Class aggregation: Sum similarities for each class
    
    Forward pass:
    1. z_support = f_phi(support) - embed support images
    2. z_query = f_phi(query) - embed query images
    3. s_i = exp(-||z_query - z_support_i||²) - compute similarities
    4. score(c) = sum(s_i for class c) - aggregate by class
    5. logits = score(c) - class scores
    """
    def __init__(self, encoder=None, embedding_dim=128):
        super().__init__()
        
        if encoder is None:
            self.encoder = SiameseEncoder(out_dim=embedding_dim)
        else:
            self.encoder = encoder
        
        self.embedding_dim = embedding_dim

    def forward_pair(self, x1, x2):
        """
        Forward pass for a pair of images.
        
        Args:
            x1: [batch_size, 3, H, W] first image
            x2: [batch_size, 3, H, W] second image
        
        Returns:
            emb1, emb2, distances
        """
        emb1 = self.encoder(x1)
        emb2 = self.encoder(x2)
        
        # Squared Euclidean distance
        diff = emb1 - emb2
        dist = torch.sum(diff ** 2, dim=1)
        
        return emb1, emb2, dist

    def compute_similarity(self, query_emb, support_embs, support_labels):
        """
        Compute similarity between query and support set for classification.
        
        Mathematical formulation:
        - Distance: d_i = ||z_q - z_s^i||²
        - Similarity: s_i = exp(-d_i)
        - Class score: score(c) = sum_{i: y_s^i = c} s_i
        
        Args:
            query_emb: [Q, D] query embeddings
            support_embs: [S, D] support embeddings
            support_labels: [S] support labels
        
        Returns:
            logits: [Q, N] class scores per query
            similarities: [Q, S] pairwise similarities
        """
        n_query = query_emb.size(0)
        n_way = len(torch.unique(support_labels))
        
        # Compute pairwise distances [Q, S]
        diff = query_emb.unsqueeze(1) - support_embs.unsqueeze(0)  # [Q, S, D]
        dists = torch.sum(diff ** 2, dim=2)  # [Q, S]
        
        # Convert to similarities using exp(-d)
        similarities = torch.exp(-dists)  # [Q, S]
        
        # Aggregate similarities by class
        unique_labels = torch.unique(support_labels)
        logits = torch.zeros(n_query, n_way, device=query_emb.device)
        
        for idx, c in enumerate(unique_labels):
            mask = (support_labels == c).unsqueeze(0)  # [1, S]
            class_sim = (similarities * mask.float()).sum(dim=1)  # [Q]
            logits[:, idx] = class_sim
        
        return logits, similarities

    def forward(self, support, support_labels, query):
        """
        Forward pass for few-shot classification.
        
        Args:
            support: Support images [S, 3, H, W]
            support_labels: Support labels [S]
            query: Query images [Q, 3, H, W]
        
        Returns:
            logits: Classification logits [Q, N]
            query_emb: Query embeddings [Q, D]
            support_emb: Support embeddings [S, D]
            similarities: [Q, S] pairwise similarities
        """
        # Encode all images
        support_emb = self.encoder(support)
        query_emb = self.encoder(query)
        
        # Compute classification logits via similarity
        logits, similarities = self.compute_similarity(query_emb, support_emb, support_labels)
        
        return logits, query_emb, support_emb, similarities


class ContrastiveLoss(nn.Module):
    """
    Contrastive loss for Siamese Network training.
    
    Mathematical formulation:
    L = (1-Y) * D² + Y * max(0, margin - D)²
    
    where:
    - Y = 1 if pair is similar (same class), 0 if dissimilar
    - D = ||f(x1) - f(x2)||² is the squared Euclidean distance
    """
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
    
    def forward(self, distance, label):
        # Similar pairs: minimize distance
        similar_loss = label * distance
        
        # Dissimilar pairs: enforce margin
        dissimilar_loss = (1 - label) * torch.clamp(
            self.margin - torch.sqrt(distance + 1e-8), min=0
        ) ** 2
        
        loss = (similar_loss + dissimilar_loss).mean()
        return loss


class PairwiseCrossEntropyLoss(nn.Module):
    """Cross-entropy loss for N-way classification using pairwise similarities."""
    
    def forward(self, logits, labels):
        return F.cross_entropy(logits, labels)


def euclidean_dist(x, y):
    """
    Compute squared Euclidean distance between two sets of vectors.
    
    Mathematical formula:
    d(x_i, y_j) = ||x_i - y_j||² = ||x_i||² + ||y_j||² - 2 * x_i · y_j
    """
    n, m, d = x.size(0), y.size(0), x.size(1)
    
    xx = (x ** 2).sum(dim=1, keepdim=True).expand(n, m)
    yy = (y ** 2).sum(dim=1, keepdim=True).expand(m, n).t()
    dist = xx + yy - 2.0 * x @ y.t()
    dist = torch.clamp(dist, min=0.0)
    
    return dist

In [ ]:
# Initialize model
model = SiameseNetwork(embedding_dim=EMBEDDING_DIM).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"{'='*60}")
print(f"Model Architecture")
print(f"{'='*60}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Embedding dimension: {EMBEDDING_DIM}")
print(f"{'='*60}\n")

# Print model summary
print("Encoder Architecture:")
print("-" * 40)
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
output = model.encoder(dummy_input)
print(f"Input shape: [1, 3, {IMAGE_SIZE}, {IMAGE_SIZE}]")
print(f"Output embedding shape: {output.shape}")
print("-" * 40)

## 6. Evaluation Metrics

### 6.1 Mathematical Formulations

**Accuracy:**
$$\text{Accuracy} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}$$

**F1-Score (Macro):**
$$F1_{\text{macro}} = \frac{1}{C} \sum_{c=1}^{C} F1_c$$

**Expected Calibration Error (ECE):**
$$ECE = \sum_{b=1}^{B} \frac{|B_b|}{n} |\text{acc}(B_b) - \text{conf}(B_b)|$$

**Attribution Sparsity:**
$$Sparsity = \frac{1}{H \cdot W} \sum_{i,j} \mathbb{1}(|a_{ij}| < \tau)$$

where $\tau = 0.6$ is the sparsity threshold.

In [ ]:
# ============================================================
# METRICS COMPUTATION
# ============================================================

def compute_ece(probs, labels, n_bins=15):
    """
    Compute Expected Calibration Error (ECE).
    
    ECE measures the difference between confidence and accuracy
    across different confidence bins.
    """
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(labels)
    
    ece = torch.zeros(1, device=probs.device)
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    
    for i in range(n_bins):
        in_bin = confidences.gt(bin_boundaries[i]) & confidences.le(bin_boundaries[i + 1])
        prop_in_bin = in_bin.float().mean()
        
        if prop_in_bin.item() > 0:
            accuracy_in_bin = accuracies[in_bin].float().mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += torch.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    
    return ece.item()


def compute_attribution_sparsity(attributions, threshold=0.6):
    """
    Compute attribution sparsity of XAI explanation maps.
    
    Higher sparsity indicates more focused explanations.
    """
    attr = np.abs(attributions)
    if attr.max() > 0:
        attr = attr / attr.max()
    
    sparsity = np.mean(attr < threshold)
    return float(sparsity)


def compute_all_metrics(y_true, y_pred, y_prob, class_names=None):
    """
    Compute comprehensive evaluation metrics.
    """
    if isinstance(y_prob, torch.Tensor):
        y_prob_np = y_prob.cpu().numpy()
    else:
        y_prob_np = y_prob
    
    # Accuracy
    acc = accuracy_score(y_true, y_pred)
    
    # F1-Scores
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    # Per-class metrics
    precision, recall, f1_per_class, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    
    # ECE
    ece_val = compute_ece(
        torch.from_numpy(y_prob_np), 
        torch.from_numpy(y_true), 
        n_bins=15
    )
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    metrics = {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_micro': f1_micro,
        'f1_weighted': f1_weighted,
        'precision_per_class': precision.tolist(),
        'recall_per_class': recall.tolist(),
        'f1_per_class': f1_per_class.tolist(),
        'support_per_class': support.tolist(),
        'ece': ece_val,
        'confusion_matrix': cm.tolist(),
    }
    
    return metrics


def print_metrics(metrics, class_names=None):
    """Pretty print metrics."""
    print(f"\n{'='*60}")
    print("EVALUATION METRICS")
    print(f"{'='*60}")
    print(f"Overall Accuracy:  {metrics['accuracy']:.4f}")
    print(f"F1-Score (Macro):  {metrics['f1_macro']:.4f}")
    print(f"F1-Score (Micro):  {metrics['f1_micro']:.4f}")
    print(f"F1-Score (Weighted): {metrics['f1_weighted']:.4f}")
    print(f"ECE (Calibration): {metrics['ece']:.4f}")
    print(f"\nPer-Class F1 Scores:")
    
    if class_names:
        for i, (name, f1) in enumerate(zip(class_names, metrics['f1_per_class'])):
            print(f"  Class {i} ({name[:15]:15s}): {f1:.4f}")
    else:
        for i, f1 in enumerate(metrics['f1_per_class']):
            print(f"  Class {i}: {f1:.4f}")
    
    print(f"{'='*60}\n")

## 7. Training Pipeline

### 7.1 Episode-Based Training

**Training Algorithm:**

```
FOR each epoch:
    FOR each episode:
        1. Sample N classes from training set
        2. Sample K support and Q query images per class
        3. Encode images: z = f_phi(images)
        4. Compute similarities: s_i = exp(-d(z_q, z_s^i))
        5. Aggregate similarities: score(c) = sum(s_i for class c)
        6. Compute loss: L = CrossEntropy(scores, y_query)
        7. Update parameters: theta = theta - lr * grad(L)
    END
END
```

In [ ]:
# ============================================================
# TRAINING FUNCTIONS
# ============================================================

def run_siamese_episode(model, optimizer, dataset, support_idx, query_idx, criterion):
    """
    Run a single few-shot episode for Siamese network training.
    """
    model.train()
    
    # Load data
    support_images = torch.stack([dataset[i][0] for i in support_idx]).to(DEVICE)
    support_labels = torch.tensor(
        [dataset[i][1] for i in support_idx], 
        dtype=torch.long
    ).to(DEVICE)
    query_images = torch.stack([dataset[i][0] for i in query_idx]).to(DEVICE)
    query_labels = torch.tensor(
        [dataset[i][1] for i in query_idx], 
        dtype=torch.long
    ).to(DEVICE)
    
    # Map labels to 0..N-1 for current episode
    unique = torch.unique(support_labels)
    label_map = {int(c): i for i, c in enumerate(unique)}
    support_labels_mapped = torch.tensor(
        [label_map[int(l)] for l in support_labels], 
        dtype=torch.long
    ).to(DEVICE)
    query_labels_mapped = torch.tensor(
        [label_map[int(l)] for l in query_labels], 
        dtype=torch.long
    ).to(DEVICE)
    
    # Forward pass
    logits, query_emb, support_emb, similarities = model(
        support_images, support_labels_mapped, query_images
    )
    
    # Compute loss
    loss = criterion(logits, query_labels_mapped)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Compute metrics
    preds = torch.argmax(logits, dim=1)
    acc = (preds == query_labels_mapped).float().mean().item()
    probs = F.softmax(logits, dim=1).detach().cpu().numpy()
    
    return (
        loss.item(), acc, 
        preds.detach().cpu().numpy(), 
        query_labels_mapped.detach().cpu().numpy(),
        probs
    )


def validate_siamese_episode(model, dataset, support_idx, query_idx, criterion):
    """Run a single few-shot episode for validation (no gradient updates)."""
    model.eval()
    
    with torch.no_grad():
        support_images = torch.stack([dataset[i][0] for i in support_idx]).to(DEVICE)
        support_labels = torch.tensor(
            [dataset[i][1] for i in support_idx], 
            dtype=torch.long
        ).to(DEVICE)
        query_images = torch.stack([dataset[i][0] for i in query_idx]).to(DEVICE)
        query_labels = torch.tensor(
            [dataset[i][1] for i in query_idx], 
            dtype=torch.long
        ).to(DEVICE)
        
        # Map labels
        unique = torch.unique(support_labels)
        label_map = {int(c): i for i, c in enumerate(unique)}
        support_labels_mapped = torch.tensor(
            [label_map[int(l)] for l in support_labels], 
            dtype=torch.long
        ).to(DEVICE)
        query_labels_mapped = torch.tensor(
            [label_map[int(l)] for l in query_labels], 
            dtype=torch.long
        ).to(DEVICE)
        
        # Forward pass
        logits, query_emb, support_emb, similarities = model(
            support_images, support_labels_mapped, query_images
        )
        
        # Compute loss
        loss = criterion(logits, query_labels_mapped)
        
        # Metrics
        preds = torch.argmax(logits, dim=1)
        acc = (preds == query_labels_mapped).float().mean().item()
        probs = F.softmax(logits, dim=1).cpu().numpy()
        
        return (loss.item(), acc, preds.cpu().numpy(), query_labels_mapped.cpu().numpy(), probs)


def train_siamese_network(model, df_train, df_val, n_epochs=NUM_EPOCHS, lr=LEARNING_RATE):
    """
    Complete training loop for Siamese Network.
    """
    train_dataset, train_sampler = pair_loader(
        df_train, train_transform, episodes=EPISODES_PER_EPOCH
    )
    val_dataset, val_sampler = pair_loader(
        df_val, eval_transform, episodes=VAL_EPISODES, 
        k_shot=K_SHOT, q_query=11
    )
    
    criterion = PairwiseCrossEntropyLoss()
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'epoch_times': []
    }
    
    best_val_acc = 0.0
    best_ckpt = None
    
    print(f"\n{'='*60}")
    print("TRAINING SIAMESE NETWORK")
    print(f"{'='*60}")
    print(f"Epochs: {n_epochs}, Episodes/Epoch: {EPISODES_PER_EPOCH}")
    print(f"Learning Rate: {lr}, Weight Decay: {WEIGHT_DECAY}")
    print(f"{'='*60}\n")
    
    epoch_start = time.time()
    
    for epoch in range(1, n_epochs + 1):
        # Training
        model.train()
        train_losses, train_accs = [], []
        
        for support_idx, query_idx in train_sampler:
            loss, acc, _, _, _ = run_siamese_episode(
                model, optimizer, train_dataset, support_idx, query_idx, criterion
            )
            train_losses.append(loss)
            train_accs.append(acc)
        
        scheduler.step()
        
        # Validation
        model.eval()
        val_losses, val_accs = [], []
        
        with torch.no_grad():
            for support_idx, query_idx in val_sampler:
                loss, acc, _, _, _ = validate_siamese_episode(
                    model, val_dataset, support_idx, query_idx, criterion
                )
                val_losses.append(loss)
                val_accs.append(acc)
        
        train_loss = float(np.mean(train_losses))
        train_acc = float(np.mean(train_accs))
        val_loss = float(np.mean(val_losses))
        val_acc = float(np.mean(val_accs))
        epoch_time = time.time() - epoch_start
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['epoch_times'].append(epoch_time)
        
        if epoch % 2 == 0:
            print(
                f"Epoch {epoch:3d}/{n_epochs} | "
                f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.3f} | "
                f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.3f} | "
                f"Time: {epoch_time:.1f}s"
            )
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_ckpt = os.path.join(CKPT_DIR, 'best_siamese.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
            }, best_ckpt)
        
        epoch_start = time.time()
    
    print(f"\nTraining Complete!")
    print(f"Best Validation Accuracy: {best_val_acc:.4f}")
    
    return history, best_ckpt

## 8. Model Evaluation

### 8.1 Test Set Evaluation

In [ ]:
# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_siamese_network(model, df_test, episodes=TEST_EPISODES):
    """
    Evaluate Siamese network on test set with episodic sampling.
    """
    test_dataset, test_sampler = pair_loader(
        df_test, eval_transform, 
        episodes=episodes, 
        k_shot=K_SHOT, 
        q_query=11
    )
    
    criterion = PairwiseCrossEntropyLoss()
    
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    all_loss, all_acc = [], []
    
    with torch.no_grad():
        for support_idx, query_idx in test_sampler:
            support_images = torch.stack([test_dataset[i][0] for i in support_idx]).to(DEVICE)
            support_labels = torch.tensor([test_dataset[i][1] for i in support_idx], dtype=torch.long).to(DEVICE)
            query_images = torch.stack([test_dataset[i][0] for i in query_idx]).to(DEVICE)
            query_labels = torch.tensor([test_dataset[i][1] for i in query_idx], dtype=torch.long).to(DEVICE)
            
            # Map labels
            unique = torch.unique(support_labels)
            label_map = {int(c): i for i, c in enumerate(unique)}
            support_labels_mapped = torch.tensor([label_map[int(l)] for l in support_labels], dtype=torch.long).to(DEVICE)
            query_labels_mapped = torch.tensor([label_map[int(l)] for l in query_labels], dtype=torch.long).to(DEVICE)
            
            # Forward pass
            logits, query_emb, support_emb, similarities = model(support_images, support_labels_mapped, query_images)
            
            loss = criterion(logits, query_labels_mapped)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            probs = F.softmax(logits, dim=1).cpu().numpy()
            
            all_loss.append(loss.item())
            all_acc.append((preds == query_labels_mapped.cpu().numpy()).mean())
            
            y_true.extend(query_labels_mapped.cpu().numpy().tolist())
            y_pred.extend(preds.tolist())
            y_prob.extend(probs.tolist())
    
    metrics = compute_all_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob), CLASS_NAMES)
    metrics['test_loss'] = float(np.mean(all_loss))
    metrics['test_acc'] = float(np.mean(all_acc))
    
    return metrics

## 9. XAI Integration

### 9.1 XAI Methods

We integrate two XAI techniques for interpreting Siamese network predictions:

**1. Grad-CAM (Gradient-weighted Class Activation Mapping)**

Grad-CAM visualizes important regions by computing gradients w.r.t. the target class score.

**2. Saliency Maps (Gradient-based)**

$$S_{ij} = \left| \frac{\partial y^c}{\partial x_{ij}} \right|$$

In [ ]:
# ============================================================
# XAI METHODS FOR SIAMESE NETWORKS
# ============================================================

class SiameseGradCAM:
    """
    Grad-CAM for Siamese Networks.
    
    Computes gradients w.r.t. similarity to support set for visualization.
    """
    def __init__(self, model, target_layer, support_images=None, support_labels=None):
        self.model = model
        self.target_layer = target_layer
        self.support_images = support_images
        self.support_labels = support_labels
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()
    
    def _register_hooks(self):
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        
        def forward_hook(module, inp, out):
            self.activations = out.detach()
        
        self.hook_handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.hook_handles.append(self.target_layer.register_full_backward_hook(backward_hook))
    
    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        self.model.zero_grad()
        
        if self.support_images is None or self.support_labels is None:
            raise ValueError("GradCAM requires support_images and support_labels")
        
        query_img = input_tensor.unsqueeze(0)
        logits, query_emb, support_emb, similarities = self.model(
            self.support_images, self.support_labels, query_img
        )
        
        if target_class is None:
            target_class = torch.argmax(logits, dim=1).item()
        
        if target_class >= logits.shape[1]:
            target_class = torch.argmax(logits, dim=1).item()
        
        score = logits[0, target_class]
        score.backward(retain_graph=True)
        
        grads = self.gradients[0]
        acts = self.activations[0]
        weights = torch.mean(grads, dim=(1, 2), keepdim=True)
        cam = torch.sum(weights * acts, dim=0).cpu().numpy()
        cam = np.maximum(cam, 0)
        cam = cam - np.min(cam)
        if np.max(cam) > 0:
            cam = cam / np.max(cam)
        
        return cam
    
    def close(self):
        for handle in self.hook_handles:
            handle.remove()


def siamese_saliency_map(model, input_tensor, support_images=None, support_labels=None, target_class=None):
    """
    Compute gradient-based saliency map for Siamese network.
    """
    model.eval()
    input_tensor = input_tensor.unsqueeze(0).clone().detach().requires_grad_(True)
    
    if support_images is None or support_labels is None:
        raise ValueError("saliency_map requires support_images and support_labels")
    
    logits, query_emb, support_emb, similarities = model(
        support_images, support_labels, input_tensor
    )
    
    if target_class is None:
        target_class = torch.argmax(logits, dim=1).item()
    
    if target_class >= logits.shape[1]:
        target_class = torch.argmax(logits, dim=1).item()
    
    score = logits[0, target_class]
    score.backward()
    
    saliency = input_tensor.grad.data.abs().squeeze().cpu().numpy()
    saliency = np.max(saliency, axis=0)
    saliency = saliency - saliency.min()
    if saliency.max() > 0:
        saliency = saliency / saliency.max()
    
    return saliency


def save_heatmap(img, mask, path, alpha=0.5, title=None):
    """Save heatmap overlay visualization."""
    img_np = img.cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_p = np.clip((img_np * std + mean), 0, 1)
    
    cmap = plt.get_cmap('jet')
    heatmap = cmap(mask)[..., :3]
    overlay = np.clip((1 - alpha) * img_p + alpha * heatmap, 0, 1)
    
    plt.figure(figsize=(5, 5))
    plt.axis('off')
    if title:
        plt.title(title)
    plt.imshow(overlay)
    plt.tight_layout(pad=0)
    plt.savefig(path, dpi=150, bbox_inches='tight', pad_inches=0.1, 
                facecolor='white', edgecolor='none')
    plt.close()

## 10. Visualization Functions

### 10.1 Training History Plots

In [ ]:
# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def plot_training_history(history, save_path=None):
    """Plot training and validation loss/accuracy curves."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss curves
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy curves
    axes[0, 1].plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
    axes[0, 1].plot(epochs, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title('Training and Validation Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Smoothed learning curves
    window = min(5, len(history['train_loss']))
    train_loss_smooth = np.convolve(history['train_loss'], np.ones(window)/window, mode='valid')
    val_loss_smooth = np.convolve(history['val_loss'], np.ones(window)/window, mode='valid')
    axes[1, 0].plot(train_loss_smooth, 'b-', label='Train Loss (smoothed)', linewidth=2)
    axes[1, 0].plot(val_loss_smooth, 'r-', label='Val Loss (smoothed)', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].set_title(f'Learning Curves (Moving Average, window={window})')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Epoch timing
    axes[1, 1].bar(epochs, history['epoch_times'], color='green', alpha=0.7)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Time (seconds)')
    axes[1, 1].set_title('Training Time per Epoch')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


def plot_confusion_matrix(cm, class_names, save_path=None, normalize=True):
    """Plot confusion matrix with per-class labels."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    if normalize:
        cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)
        fmt = '.2%'
    else:
        cm_normalized = cm
        fmt = 'd'
    
    sns.heatmap(cm_normalized, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


def plot_per_class_metrics(metrics, class_names, save_path=None):
    """Plot per-class precision, recall, and F1-score."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    x = np.arange(len(class_names))
    width = 0.25
    
    axes[0].bar(x, metrics['precision_per_class'], width, label='Precision', color='steelblue')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Score')
    axes[0].set_title('Per-Class Precision')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([c[:10] for c in class_names], rotation=45, ha='right')
    axes[0].set_ylim([0, 1])
    axes[0].grid(True, alpha=0.3, axis='y')
    
    axes[1].bar(x, metrics['recall_per_class'], width, label='Recall', color='forestgreen')
    axes[1].set_xlabel('Class')
    axes[1].set_ylabel('Score')
    axes[1].set_title('Per-Class Recall')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([c[:10] for c in class_names], rotation=45, ha='right')
    axes[1].set_ylim([0, 1])
    axes[1].grid(True, alpha=0.3, axis='y')
    
    axes[2].bar(x, metrics['f1_per_class'], width, label='F1-Score', color='coral')
    axes[2].set_xlabel('Class')
    axes[2].set_ylabel('Score')
    axes[2].set_title('Per-Class F1-Score')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels([c[:10] for c in class_names], rotation=45, ha='right')
    axes[2].set_ylim([0, 1])
    axes[2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

## 11. XAI Visualization Generation

In [ ]:
# ============================================================
# XAI VISUALIZATION
# ============================================================

def generate_xai_explanations(model, df_test, n_samples=5, save_dir=XAI_DIR):
    """Generate XAI explanations for test samples."""
    ds = ImagePathsDataset(df_test, transform=eval_transform)
    sampler = SiameseSampler(df_test['label'].to_numpy(), n_way=N_WAY, k_shot=K_SHOT, q_query=11, episodes=1)
    
    support_idx, query_idx = next(iter(sampler))
    
    support_images = torch.stack([ds[i][0] for i in support_idx]).to(DEVICE)
    support_labels = torch.tensor([ds[i][1] for i in support_idx], dtype=torch.long).to(DEVICE)
    
    unique = torch.unique(support_labels)
    label_map = {int(c): i for i, c in enumerate(unique)}
    support_labels_mapped = torch.tensor([label_map[int(l)] for l in support_labels], dtype=torch.long).to(DEVICE)
    
    query_images = torch.stack([ds[i][0] for i in query_idx]).to(DEVICE)
    query_labels = torch.tensor([ds[i][1] for i in query_idx], dtype=torch.long).to(DEVICE)
    
    print(f"\n{'='*60}")
    print("XAI VISUALIZATION GENERATION (Siamese Network)")
    print(f"{'='*60}")
    
    sparsity_results = []
    
    for idx in range(min(n_samples, len(query_idx))):
        img = query_images[idx]
        true_label = query_labels[idx].item()
        
        logits, query_emb, support_emb, similarities = model(
            support_images, support_labels_mapped, img.unsqueeze(0)
        )
        pred = torch.argmax(logits, dim=1).item()
        confidence = F.softmax(logits, dim=1)[0, pred].item()
        pred_original = int(list(label_map.keys())[list(label_map.values()).index(pred)])
        
        # Target layer for Grad-CAM
        target_layer = model.encoder.conv_blocks[4]
        
        gradcam = SiameseGradCAM(model, target_layer=target_layer,
                                support_images=support_images, support_labels=support_labels_mapped)
        cam_mask = gradcam.generate(img, target_class=pred)
        gradcam.close()
        
        sal_map = siamese_saliency_map(model, img, support_images=support_images,
                                       support_labels=support_labels_mapped, target_class=pred)
        
        sparsity_cam = compute_attribution_sparsity(cam_mask)
        sparsity_sal = compute_attribution_sparsity(sal_map)
        
        sparsity_results.append({
            'sample': idx,
            'true_label': CLASS_NAMES[true_label] if true_label < len(CLASS_NAMES) else f'Class {true_label}',
            'pred_label': CLASS_NAMES[pred_original] if pred_original < len(CLASS_NAMES) else f'Class {pred_original}',
            'confidence': confidence,
            'sparsity_cam': sparsity_cam,
            'sparsity_sal': sparsity_sal
        })
        
        save_heatmap(img.cpu(), cam_mask, 
                     os.path.join(save_dir, f'siamese_gradcam_sample{idx}_true{true_label}_pred{pred_original}.png'),
                     title=f'Siamese Grad-CAM: True={true_label}, Pred={pred_original}')
        
        save_heatmap(img.cpu(), sal_map, 
                     os.path.join(save_dir, f'siamese_saliency_sample{idx}_true{true_label}_pred{pred_original}.png'),
                     title=f'Siamese Saliency: True={true_label}, Pred={pred_original}')
        
        print(f"Sample {idx}: True={true_label}, Pred={pred_original}, "
              f"Conf={confidence:.3f}, CAM sparsity={sparsity_cam:.3f}, Saliency sparsity={sparsity_sal:.3f}")
    
    return sparsity_results

## 12. Statistical Significance Testing

### 12.1 T-Test for Comparing Model Runs

In [ ]:
# ============================================================
# STATISTICAL SIGNIFICANCE TESTING
# ============================================================

def run_multiple_experiments(n_runs=3, epochs_per_run=None):
    """Run multiple independent experiments for statistical analysis."""
    if epochs_per_run is None:
        epochs_per_run = max(5, NUM_EPOCHS // 3)
    
    print(f"\n{'='*60}")
    print(f"MULTIPLE EXPERIMENT RUNS")
    print(f"{'='*60}")
    print(f"Number of runs: {n_runs}")
    print(f"Epochs per run: {epochs_per_run}")
    print(f"{'='*60}\n")
    
    results = []
    
    for run in range(1, n_runs + 1):
        print(f"\n--- Run {run}/{n_runs} ---")
        
        seed = RNG_SEED + run
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
        
        df_train_r, df_val_r, df_test_r = make_stratified_splits(DATA_ROOT, seed=seed)
        
        model = SiameseNetwork(embedding_dim=EMBEDDING_DIM).to(DEVICE)
        
        history, ckpt = train_siamese_network(model, df_train_r, df_val_r, n_epochs=epochs_per_run, lr=LEARNING_RATE)
        model.load_state_dict(torch.load(ckpt)['model_state'])
        
        metrics = evaluate_siamese_network(model, df_test_r)
        metrics['run'] = run
        results.append(metrics)
        
        print(f"  Run {run} Test Accuracy: {metrics['accuracy']:.4f}")
    
    accuracies = [r['accuracy'] for r in results]
    f1_scores = [r['f1_macro'] for r in results]
    
    t_acc, p_acc = ttest_ind([accuracies[0]], accuracies[1:] if len(accuracies) > 1 else accuracies)
    t_f1, p_f1 = ttest_ind([f1_scores[0]], f1_scores[1:] if len(f1_scores) > 1 else f1_scores)
    
    stats = {
        'n_runs': n_runs,
        'accuracies': accuracies,
        'f1_scores': f1_scores,
        'mean_accuracy': np.mean(accuracies),
        'std_accuracy': np.std(accuracies),
        'mean_f1': np.mean(f1_scores),
        'std_f1': np.std(f1_scores),
        't_test_accuracy': {'t_stat': float(t_acc), 'p_val': float(p_acc)},
        't_test_f1': {'t_stat': float(t_f1), 'p_val': float(p_f1)}
    }
    
    print(f"\n{'='*60}")
    print("STATISTICAL ANALYSIS")
    print(f"{'='*60}")
    print(f"Accuracy: {stats['mean_accuracy']:.4f} ± {stats['std_accuracy']:.4f}")
    print(f"F1-Score: {stats['mean_f1']:.4f} ± {stats['std_f1']:.4f}")
    print(f"T-test (Accuracy): t={t_acc:.4f}, p={p_acc:.4f}")
    print(f"T-test (F1): t={t_f1:.4f}, p={p_f1:.4f}")
    print(f"{'='*60}\n")
    
    return results, stats

## 13. Main Execution Pipeline

In [ ]:
# ============================================================
# MAIN EXECUTION
# ============================================================

def main():
    """Execute the complete Siamese network few-shot learning pipeline."""
    print(f"\n{'='*70}")
    print(f"SIAMESE NETWORKS FOR FEW-SHOT LEARNING WITH XAI")
    print(f"{'='*70}\n")
    
    start_time = time.time()
    
    # STEP 1: DATA SPLITTING
    print("\n" + "="*50)
    print("STEP 1: DATA PREPARATION")
    print("="*50)
    df_train, df_val, df_test, CLASS_NAMES = make_stratified_splits(DATA_ROOT)
    
    # Visualize class distribution
    fig, ax = plt.subplots(figsize=(10, 5))
    class_counts = df_train['class'].value_counts().sort_index()
    ax.bar(range(len(CLASS_NAMES)), class_counts.values, color='steelblue', alpha=0.8)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels([c[:15] for c in CLASS_NAMES], rotation=45, ha='right')
    ax.set_ylabel('Number of Images')
    ax.set_title('Training Set Class Distribution')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'class_distribution.png'), dpi=150)
    plt.show()
    plt.close()
    
    # STEP 2: MODEL TRAINING
    print("\n" + "="*50)
    print("STEP 2: MODEL TRAINING (Siamese Network)")
    print("="*50)
    
    model = SiameseNetwork(embedding_dim=EMBEDDING_DIM).to(DEVICE)
    print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters\n")
    
    history, best_ckpt = train_siamese_network(model, df_train, df_val, n_epochs=NUM_EPOCHS, lr=LEARNING_RATE)
    
    plot_training_history(history, save_path=os.path.join(PLOTS_DIR, 'training_history.png'))
    
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'siamese_final.pth'))
    with open(os.path.join(OUTPUT_DIR, 'train_history.json'), 'w') as f:
        json.dump(history, f, indent=2)
    
    model.load_state_dict(torch.load(best_ckpt)['model_state'])
    
    # STEP 3: MODEL EVALUATION
    print("\n" + "="*50)
    print("STEP 3: MODEL EVALUATION")
    print("="*50)
    
    test_metrics = evaluate_siamese_network(model, df_test)
    
    print_metrics(test_metrics, CLASS_NAMES)
    
    with open(os.path.join(OUTPUT_DIR, 'test_metrics.json'), 'w') as f:
        json.dump(test_metrics, f, indent=2)
    
    print("\nGenerating evaluation visualizations...")
    
    plot_confusion_matrix(np.array(test_metrics['confusion_matrix']), CLASS_NAMES,
                          save_path=os.path.join(PLOTS_DIR, 'confusion_matrix.png'), normalize=True)
    
    plot_per_class_metrics(test_metrics, CLASS_NAMES,
                           save_path=os.path.join(PLOTS_DIR, 'per_class_metrics.png'))
    
    # STEP 4: XAI VISUALIZATION
    print("\n" + "="*50)
    print("STEP 4: XAI VISUALIZATION")
    print("="*50)
    
    sparsity_results = generate_xai_explanations(model, df_test, n_samples=6)
    
    with open(os.path.join(OUTPUT_DIR, 'xai_results.json'), 'w') as f:
        json.dump(sparsity_results, f, indent=2)
    
    total_time = time.time() - start_time
    
    print(f"\n{'='*70}")
    print(f"PIPELINE COMPLETE")
    print(f"{'='*70}")
    print(f"Total execution time: {total_time/60:.1f} minutes")
    print(f"\nFinal Results:")
    print(f"  Test Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"  Test F1-Macro: {test_metrics['f1_macro']:.4f}")
    print(f"  ECE: {test_metrics['ece']:.4f}")
    print(f"\nOutputs saved to: {OUTPUT_DIR}")
    print(f"{'='*70}\n")
    
    return model, test_metrics, history

# Run main pipeline
model, test_metrics, history = main()

## 14. Final Results Summary and Conclusions

In [ ]:
# ============================================================
# FINAL RESULTS AND CONCLUSIONS
# ============================================================

# Load and display final metrics
with open(os.path.join(OUTPUT_DIR, 'test_metrics.json'), 'r') as f:
    final_metrics = json.load(f)

print(f"\n{'='*70}")
print(f"FINAL EVALUATION RESULTS")
print(f"{'='*70}")
print(f"\n{'Metric':<30} {'Value':>15}")
print(f"{'-'*45}")
print(f"{'Test Accuracy':<30} {final_metrics['accuracy']:>15.4f}")
print(f"{'Test F1-Score (Macro)':<30} {final_metrics['f1_macro']:>15.4f}")
print(f"{'Test F1-Score (Micro)':<30} {final_metrics['f1_micro']:>15.4f}")
print(f"{'Test F1-Score (Weighted)':<30} {final_metrics['f1_weighted']:>15.4f}")
print(f"{'Expected Calibration Error':<30} {final_metrics['ece']:>15.4f}")
print(f"\n{'='*70}")
print(f"\nPer-Class Performance:")
print(f"\n{'Class':<25} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"{'-'*55}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name[:25]:<25} "
          f"{final_metrics['precision_per_class'][i]:>10.4f} "
          f"{final_metrics['recall_per_class'][i]:>10.4f} "
          f"{final_metrics['f1_per_class'][i]:>10.4f}")
print(f"{'='*70}\n")

## 15. Research Contributions and Future Work

### 15.1 Summary of Contributions

This implementation demonstrates a complete Siamese network for few-shot learning with:

1. **Siamese Architecture**: Weight-sharing encoder for comparing image pairs

2. **Similarity-Based Classification**: Aggregating pairwise similarities to determine class

3. **XAI Integration**: Grad-CAM and saliency maps for interpretable predictions

4. **Comprehensive Evaluation**: Accuracy, F1-score, ECE, and attribution sparsity

5. **Statistical Validation**: T-tests across multiple runs

### 15.2 Key Mathematical Equations

- **Embedding**: $\mathbf{z} = f_\phi(\mathbf{x})$
- **Distance**: $d(\mathbf{z}_1, \mathbf{z}_2) = \|\mathbf{z}_1 - \mathbf{z}_2\|^2$
- **Similarity**: $s = \exp(-d(\mathbf{z}_1, \mathbf{z}_2))$
- **Class Score**: $\text{score}(c) = \sum_{i: y_i = c} s_i$
- **Probability**: $P(y=c|\mathbf{x}_q) = \text{softmax}(\text{score})_c$

### 15.3 References

1. Bromley, J., et al. (1994). Signature verification using a siamese time delay neural network. NeurIPS.
2. Chopra, S., Hadsell, R., & LeCun, Y. (2005). Learning a similarity metric discriminatively. CVPR.
3. Koch, G., Zemel, R., & Salakhutdinov, R. (2015). Siamese neural networks for one-shot image recognition. ICML Workshop.
4. Selvaraju, R. R., et al. (2017). Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization. ICCV.

In [ ]:
# ============================================================
# SAVE ALL RESULTS
# ============================================================

summary = {
    'experiment_config': {
        'n_way': N_WAY,
        'k_shot': K_SHOT,
        'q_query': Q_QUERY,
        'embedding_dim': EMBEDDING_DIM,
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'margin': MARGIN,
        'train_size': len(df_train),
        'val_size': len(df_val),
        'test_size': len(df_test),
    },
    'final_metrics': final_metrics,
    'class_names': CLASS_NAMES,
    'output_directory': OUTPUT_DIR
}

with open(os.path.join(OUTPUT_DIR, 'experiment_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

# List all generated files
print("\n" + "="*70)
print("GENERATED FILES")
print("="*70)

for dir_path, dir_name in [(SPLIT_DIR, 'Splits'), (PLOTS_DIR, 'Plots'), 
                            (XAI_DIR, 'XAI'), (CKPT_DIR, 'Checkpoints')]:
    print(f"\n{dir_name} Directory ({dir_path}):")
    if os.path.exists(dir_path):
        for f in sorted(os.listdir(dir_path)):
            print(f"  - {f}")

print(f"\n{'='*70}")
print("NOTEBOOK EXECUTION COMPLETE")
print(f"{'='*70}\n")